# Set-2 Banking & Loan Analytics: Q1 Complete Solution
**Run all:** `Kernel -> Restart & Run All`. Needs `Bank_Data.xlsx` (sheets: Customers, Loans, Transactions) in the same folder.

## 0. Load the data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

cust  = pd.read_excel('Bank_Data.xlsx', sheet_name='Customers')
loans = pd.read_excel('Bank_Data.xlsx', sheet_name='Loans')
trans = pd.read_excel('Bank_Data.xlsx', sheet_name='Transactions')

print('Shapes (rows, cols):', cust.shape, loans.shape, trans.shape)
for d in (cust, loans, trans):
    display(d.head(3))
    print(d.isna().sum(), '\n')
print('Duplicate rows:', cust.duplicated().sum(), loans.duplicated().sum(), trans.duplicated().sum())

## 1. Missing and duplicate values (Customer_ID, Annual_Income, Credit_Score, Transaction_Amount)

In [ ]:
# duplicates
cust  = cust.drop_duplicates()
loans = loans.drop_duplicates()
trans = trans.drop_duplicates()

# Customer_ID missing -> cannot link records, so drop
cust  = cust.dropna(subset=['Customer_ID'])
loans = loans.dropna(subset=['Customer_ID'])
trans = trans.dropna(subset=['Customer_ID'])

# numeric gaps -> median
cust['Annual_Income'] = cust['Annual_Income'].fillna(cust['Annual_Income'].median())
cust['Credit_Score']  = cust['Credit_Score'].fillna(cust['Credit_Score'].median())
trans['Transaction_Amount'] = trans['Transaction_Amount'].fillna(trans['Transaction_Amount'].median())

print(cust[['Customer_ID','Annual_Income','Credit_Score']].isna().sum())
print(trans['Transaction_Amount'].isna().sum())
print(cust.shape, loans.shape, trans.shape)

## 2. Inconsistent values (Employment_Type, Account_Type, Loan_Status, Channel)

In [ ]:
print(cust['Employment_Type'].value_counts(), '\n')
print(cust['Account_Type'].value_counts(), '\n')
print(loans['Loan_Status'].value_counts(), '\n')
print(trans['Channel'].value_counts())

In [ ]:
# Step A: strip spaces + Title Case.  Step B: replace leftovers.
cust['Employment_Type'] = cust['Employment_Type'].str.strip().str.title()
cust['Employment_Type'] = cust['Employment_Type'].replace({'Self Employed': 'Self-Employed'})

cust['Account_Type'] = cust['Account_Type'].str.strip().str.title()
cust['Account_Type'] = cust['Account_Type'].replace({'Saving': 'Savings', 'Salar': 'Salary', 'Curren': 'Current'})

loans['Loan_Status'] = loans['Loan_Status'].str.strip().str.title()

trans['Channel'] = trans['Channel'].str.strip().str.title()
trans['Channel'] = trans['Channel'].replace({'Atm': 'ATM', 'Mobile-App': 'Mobile App'})

print(cust['Employment_Type'].value_counts(), '\n')
print(cust['Account_Type'].value_counts(), '\n')
print(loans['Loan_Status'].value_counts(), '\n')
print(trans['Channel'].value_counts())

## 3. Abnormal values (Age, Credit_Score, Interest_Rate, Tenure_Years)
Rule: mark impossible values as NaN, then fill with the median.

In [ ]:
# Age: valid 18 to 100
cust.loc[(cust['Age'] < 18) | (cust['Age'] > 100), 'Age'] = np.nan
cust['Age'] = cust['Age'].fillna(cust['Age'].median())

# Credit_Score: valid 300 to 900
cust.loc[(cust['Credit_Score'] < 300) | (cust['Credit_Score'] > 900), 'Credit_Score'] = np.nan
cust['Credit_Score'] = cust['Credit_Score'].fillna(cust['Credit_Score'].median())

# Interest_Rate: valid 1 to 30 %
loans.loc[(loans['Interest_Rate'] < 1) | (loans['Interest_Rate'] > 30), 'Interest_Rate'] = np.nan
loans['Interest_Rate'] = loans['Interest_Rate'].fillna(loans['Interest_Rate'].median())

# Tenure_Years: valid 1 to 30
loans.loc[(loans['Tenure_Years'] < 1) | (loans['Tenure_Years'] > 30), 'Tenure_Years'] = np.nan
loans['Tenure_Years'] = loans['Tenure_Years'].fillna(loans['Tenure_Years'].median())

print(cust[['Age','Credit_Score']].describe(), '\n')
print(loans[['Interest_Rate','Tenure_Years']].describe())

## 4. Inappropriate values (Loan_Amount, Collateral_Value, Transaction_Amount, Account_Balance)
Step A: zero / negative are invalid -> NaN -> median.  Step B: cap huge outliers using the IQR rule.

In [ ]:
# Step A: negative / zero values are not valid
loans.loc[loans['Loan_Amount'] <= 0, 'Loan_Amount'] = np.nan
loans.loc[loans['Collateral_Value'] <= 0, 'Collateral_Value'] = np.nan
trans.loc[trans['Transaction_Amount'] <= 0, 'Transaction_Amount'] = np.nan
cust.loc[cust['Account_Balance'] < 0, 'Account_Balance'] = np.nan     # balance of 0 is allowed

loans['Loan_Amount']       = loans['Loan_Amount'].fillna(loans['Loan_Amount'].median())
loans['Collateral_Value']  = loans['Collateral_Value'].fillna(loans['Collateral_Value'].median())
trans['Transaction_Amount']= trans['Transaction_Amount'].fillna(trans['Transaction_Amount'].median())
cust['Account_Balance']    = cust['Account_Balance'].fillna(cust['Account_Balance'].median())

# Step B: cap extreme outliers (IQR rule)
def cap_outliers(df, col):
    q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
    upper = q3 + 1.5 * (q3 - q1)
    df[col] = df[col].clip(upper=upper)

cap_outliers(loans, 'Loan_Amount')
cap_outliers(loans, 'Collateral_Value')
cap_outliers(trans, 'Transaction_Amount')
cap_outliers(cust,  'Account_Balance')

print(loans[['Loan_Amount','Collateral_Value']].describe(), '\n')
print(trans['Transaction_Amount'].describe(), '\n')
print(cust['Account_Balance'].describe())

## 5. Matplotlib validation

In [ ]:
fig, ax = plt.subplots(3, 3, figsize=(16, 12))

ax[0,0].hist(cust['Age'], bins=15);                    ax[0,0].set_title('Age')
ax[0,1].hist(cust['Credit_Score'], bins=15);           ax[0,1].set_title('Credit Score')
ax[0,2].boxplot(loans['Interest_Rate']);               ax[0,2].set_title('Interest Rate')
ax[1,0].boxplot(loans['Tenure_Years']);                ax[1,0].set_title('Tenure Years')
ax[1,1].boxplot(loans['Loan_Amount']);                 ax[1,1].set_title('Loan Amount')
ax[1,2].scatter(loans['Loan_Amount'], loans['Collateral_Value'], s=8)
ax[1,2].set_title('Loan Amount vs Collateral')
ax[2,0].boxplot(trans['Transaction_Amount']);          ax[2,0].set_title('Transaction Amount')
ax[2,1].boxplot(cust['Account_Balance']);              ax[2,1].set_title('Account Balance')
loans['Loan_Status'].value_counts().plot(kind='bar', ax=ax[2,2]); ax[2,2].set_title('Loan Status')

plt.tight_layout()
plt.show()

## 6. Export cleaned CSVs (for Power BI)

In [ ]:
cust.to_csv('customers_clean.csv', index=False)
loans.to_csv('loans_clean.csv', index=False)
trans.to_csv('transactions_clean.csv', index=False)

print('Saved:', cust.shape, loans.shape, trans.shape)
print('Customer_ID unique?', cust['Customer_ID'].is_unique)

## Power BI DAX measures (Q2)
```
Total Customers = DISTINCTCOUNT(customers_clean[Customer_ID])
Total Loans = COUNTROWS(loans_clean)
Total Loan Amount = SUM(loans_clean[Loan_Amount])
Total Transaction Amount = SUM(transactions_clean[Transaction_Amount])
Average Credit Score = AVERAGE(customers_clean[Credit_Score])
```